In [1]:
import spacy
nlp = spacy.load("de_core_news_sm")

In [1]:
import pandas as pd
import pickle
import csv

# evaluation data
path_train = "../Input/Data/train/"
# task 1 file contains all info abt spans
eval_all_data = pd.read_json(path_train + "test_task1.json")
# only use flausch comments (which have gold spans)
eval_flausch_data = eval_all_data[eval_all_data["flausch"] == "yes"].copy()

# training data
train_all_data = pd.read_json(path_train + "train_task1.json")
train_flausch_data = train_all_data[train_all_data["flausch"] == "yes"].copy()

# test data
path_test = "../Input/Data/test/"
# all comments
test_all_data = pd.read_csv(path_test + "comments.csv")
# only flausch comments
test_task1 = pd.read_csv(path_test + "task1.csv")
# merge comments & task 1 flausch info
test_flausch_data = pd.merge(test_all_data, test_task1, how="inner")
# only keep flausch comments
test_flausch_data = test_flausch_data[test_flausch_data["flausch"] == "yes"]

# Subtask 1 predictions for test data
path_thesis = "Data/"
# class predictions from best configuration for subtask 1
with open(path_thesis + "subtask1_test_predictions.pkl", "rb") as infile: 
   subtask1_test_predictions = pickle.load(infile)
# new dataframe with predicted classes
test_task1_predicted = test_task1.copy()
test_task1_predicted["flausch"] = subtask1_test_predictions
# merge comments & flausch predictions
test_flausch_data_predicted = pd.merge(test_all_data, test_task1_predicted, how="inner")
# only keep predicted flausch comments
test_flausch_data_predicted = test_flausch_data_predicted[test_flausch_data_predicted["flausch"] == "yes"]

In [2]:
# adding gold spans to test_flausch_data for span_accuracy evaluation

task2 = pd.read_csv(path_test + "task2.csv")

span_pairs = []
for i in range(len(test_flausch_data)):
    span_pairs_i = []

    document = test_flausch_data.iloc[i]["document"]
    comment_id = test_flausch_data.iloc[i]["comment_id"]

    spans = task2[(task2["document"] == document) & (task2["comment_id"] == comment_id)]
    for index, row in spans.iterrows():
        span_pairs_i.append([row["start"], row["end"]])
    
    span_pairs.append(span_pairs_i)

test_flausch_data["span_pairs"] = span_pairs

In [ ]:
# span detection model trained on original spaCy spans

from transformers import pipeline

pipe = pipeline("text-classification", model="Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan")

# Evaluation functions

In [ ]:
# evaluation for span accuracy

def span_accuracy(data):
    
    scores = {"exact matches": 0, "correct start": 0, "correct end": 0}

    for i in range(len(data)):

        exact_matches = 0
        correct_start = 0
        correct_end = 0

        for span in data.iloc[i]["span_pairs"]:

            dependency_spans = data.iloc[i]["dependency_spans"]

            # entire span matches
            if span in dependency_spans:
                exact_matches += 1
                correct_start += 1
                correct_end += 1
            # only span start matches
            elif span[0] in [s[0] for s in dependency_spans]:
                correct_start += 1
            # only span end matches
            elif span[1] in [s[1] for s in dependency_spans]:
                correct_end += 1


        # average for spans in comment    
        scores["exact matches"] += exact_matches/len(data.iloc[i]["span_pairs"])
        scores["correct start"] += correct_start/len(data.iloc[i]["span_pairs"])
        scores["correct end"] += correct_end/len(data.iloc[i]["span_pairs"])

    # average for all comments
    scores["exact matches"] /= len(data)
    scores["correct start"] /= len(data)
    scores["correct end"] /= len(data)

    return scores

In [4]:
# official evaluation for span classification

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from multiset import *

# Labels for the fine grained classification
ALL_LABELS = ["affection declaration","agreement","ambiguous",
              "compliment","encouragement","gratitude","group membership",
              "implicit","positive feedback","sympathy"]

# TASK 1
def binary_eval(file_gold, file_pred):

    test_g = pd.read_csv(file_gold)
    test_p = pd.read_csv(file_pred)

    (bin_prec, bin_rec, bin_f, bin_supp) = precision_recall_fscore_support(test_g.flausch, test_p.flausch, pos_label="yes", average='binary')

    print("TASK 1: BINARY CLASSIFICATION",
          "\n=============================",
          "\nPrecision:\t %.4f" % bin_prec,
          "\nRecall:\t\t %.4f" % bin_rec,
          "\nF-score:\t %.4f" % bin_f)

    return((bin_prec, bin_rec, bin_f))


def fine_grained_flausch_by_label(file_gold, file_pred):

    # read files
    gold = pd.read_csv(file_gold)
    gold['cid']= gold['document']+"_"+gold['comment_id'].apply(str)
    predicted = pd.read_csv(file_pred)
    predicted['cid']= predicted['document']+"_"+predicted['comment_id'].apply(str)


    # annotation sets (predicted)
    pred_spans = Multiset()
    pred_spans_loose = Multiset()
    pred_types = Multiset()

    # annotation sets (gold)
    gold_spans = Multiset()
    gold_spans_loose = Multiset()
    gold_types = Multiset()

    for row in predicted.itertuples(index=False):
        pred_spans.add((row.cid,row.type,row.start,row.end))
        pred_spans_loose.add((row.cid,row.start,row.end))
        pred_types.add((row.cid,row.type))
    for row in gold.itertuples(index=False):
        gold_spans.add((row.cid,row.type,row.start,row.end))
        gold_spans_loose.add((row.cid,row.start,row.end))
        gold_types.add((row.cid,row.type))

    # precision = true_pos / true_pos + false_pos
    # recall = true_pos / true_pos + false_neg
    # f_1 = 2 * prec * rec / (prec + rec)

    results = {'TOTAL': {'STRICT': {},'SPANS': {},'TYPES': {}}}
    # label-wise evaluation (only for strict and type)
    for label in ALL_LABELS:
        results[label] = {'STRICT': {},'TYPES': {}}
        gold_spans_x = set(filter(lambda x: x[1].__eq__(label), gold_spans))
        pred_spans_x = set(filter(lambda x: x[1].__eq__(label), pred_spans))
        gold_types_x = set(filter(lambda x: x[1].__eq__(label), gold_types))
        pred_types_x = set(filter(lambda x: x[1].__eq__(label), pred_types))

        # strict: spans + type must match
        ### NOTE: x and y / x returns 0 if x = 0 and y/x otherwise (test for zero division)
        strict_p = float(len(pred_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(pred_spans_x)
        strict_r = float(len(gold_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(gold_spans_x)
        strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
        results[label]['STRICT']['prec'] = strict_p
        results[label]['STRICT']['rec'] = strict_r
        results[label]['STRICT']['f1'] = strict_f

        # detection mode: only types must match (per post)
        types_p = float(len(pred_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(pred_types_x)
        types_r = float(len(gold_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(gold_types_x)
        types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
        results[label]['TYPES']['prec'] = types_p
        results[label]['TYPES']['rec'] = types_r
        results[label]['TYPES']['f1'] = types_f

    # Overall evaluation
    # strict: spans + type must match
    strict_p = float(len(pred_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(pred_spans)
    strict_r = float(len(gold_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(gold_spans)
    strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
    results['TOTAL']['STRICT']['prec'] = strict_p
    results['TOTAL']['STRICT']['rec'] = strict_r
    results['TOTAL']['STRICT']['f1'] = strict_f

    # spans: spans must match
    spans_p = float(len(pred_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(pred_spans_loose)
    spans_r = float(len(gold_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(gold_spans_loose)
    spans_f = (spans_p + spans_r) and 2 * spans_p * spans_r / (spans_p + spans_r)
    results['TOTAL']['SPANS']['prec'] = spans_p
    results['TOTAL']['SPANS']['rec'] = spans_r
    results['TOTAL']['SPANS']['f1'] = spans_f

    # detection mode: only types must match (per post)
    types_p = float(len(pred_types)) and float( len(gold_types.intersection(pred_types))) / len(pred_types)
    types_r = float(len(gold_types)) and float( len(gold_types.intersection(pred_types))) / len(gold_types)
    types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
    results['TOTAL']['TYPES']['prec'] = types_p
    results['TOTAL']['TYPES']['rec'] = types_r
    results['TOTAL']['TYPES']['f1'] = types_f

    print("STRICT:\n ",strict_p,strict_r,strict_f)
    print("SPANS:\n ",spans_p,spans_r,spans_f)
    print("TYPES:\n ",types_p,types_r,types_f)
    return(results)

# Original configuration

In [5]:
def root(token):
    if token.dep_ in ["ROOT", "rs", "cd", "ju"]:
        return token
    else:
        return root(token.head)

In [ ]:
import copy

def words_to_indices(comment, word_spans):

    index_spans = copy.deepcopy(word_spans)

    spans_id = 0
    span_id = 0

    for i in range(len(comment) + 1):
        if spans_id >= len(word_spans):
            index_spans[spans_id - 1][1] = i
            continue

        looking_for = word_spans[spans_id][span_id]

        if span_id == 0:
            if comment[i:].startswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to looking for end of span
                span_id = 1
        else:
            if comment[:i].endswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to beginning of next span
                spans_id += 1
                span_id = 0

                if spans_id < len(word_spans):
                    looking_for = word_spans[spans_id][span_id]

                    if comment[i:].startswith(looking_for):
                        index_spans[spans_id][span_id] = i
                        # move to looking for end of span
                        span_id = 1

    # dealing with spans that are only one character
    # & thus can't be identified
    for i in range(len(index_spans)):
        if type(index_spans[i][0]) is str:
            index_spans[i][0] = index_spans[i-1][1]
            index_spans[i][1] = index_spans[i][0]
        elif type(index_spans[i][1]) is str:
            index_spans[i][1] = index_spans[i][0]

    return index_spans


def get_spans_orig(comment):

    head = None
    last = None
    spans = []
    current_span = []

    for token in nlp(comment):

        if root(token) != head:

            if last != None: # not needed for first span
                # save end of last span
                current_span.append(last.text)
                spans.append(current_span)

            # start new span
            current_span = [token.text]
            head = root(token)
        
        last = token

    # end span for last token
    current_span.append(last.text)
    spans.append(current_span)
    
    return words_to_indices(comment, spans)

In [7]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans_orig(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.47799775058486144, 'correct start': 0.8152098622303681, 'correct end': 0.6235365174734574}


In [8]:
# save span predictions for all comments on eval set

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(eval_all_data)):
    document = eval_all_data.iloc[i]["document"]
    comment_id = eval_all_data.iloc[i]["comment_id"]
    comment = eval_all_data.iloc[i]["comment"]

    spans = get_spans_orig(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])
eval_all_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
eval_all_spans.to_csv(path_thesis + "eval_all_spans_orig.csv", index=False)

In [ ]:
# run classification pipeline on predicted spans for all cmments

eval_all_spans = pd.read_csv(path_thesis + "eval_all_spans_orig.csv")

# update "type" to predicted type for "text"
for i in range(len(eval_all_spans)):
    text = eval_all_spans.iloc[i]["text"]

    # update type (label)
    eval_all_spans.loc[i, "type"] = pipe(text)[0]["label"]

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = eval_all_spans[eval_all_spans["type"] != "not"].drop(columns=["text"])
# save predictions
pred_data.to_csv(path_thesis + "eval_all_spans_labels_orig.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for all comments

file_gold = path_train + "gold_data_task2.csv"
file_pred = path_thesis + "eval_all_spans_labels_orig.csv"
results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.39578630549285176 0.34673698088332233 0.3696416022487702
SPANS:
  0.4161023325808879 0.36453526697429134 0.3886156008432888
TYPES:
  0.7848006019563581 0.6875411997363217 0.7329585382993675
positive feedback
        STRICT     TYPES
prec  0.443787  0.825210
rec   0.369004  0.740573
f1    0.402955  0.780604

affection declaration
        STRICT     TYPES
prec  0.479070  0.822115
rec   0.408730  0.750000
f1    0.441113  0.784404

group membership
        STRICT     TYPES
prec  0.214286  0.692308
rec   0.142857  0.500000
f1    0.171429  0.580645

encouragement
        STRICT     TYPES
prec  0.235955  0.869048
rec   0.241379  0.879518
f1    0.238636  0.874251

compliment
        STRICT     TYPES
prec  0.314815  0.790123
rec   0.318352  0.810127
f1    0.316574  0.800000

ambiguous
        STRICT     TYPES
prec  0.400000  0.400000
rec   0.272727  0.272727
f1    0.324324  0.324324

implicit
        STRICT     TYPES
prec  0.250000  0.500000
rec   0.090909  0.181818
f1    0.133333  

# Improving the functions

## Altering the base function

In [ ]:
# combines original get_spans and words_to_indices, 

def get_spans1(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if root(token) != current_root:

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    
    return spans

In [10]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans1(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.4853354663175815, 'correct start': 0.8151653218528027, 'correct end': 0.6333167753033611}


## Adding rules

In [ ]:
# no "," or "." at beginning or end of span

def get_spans2(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if token.text in [",", "."]:
            # ",", ".", "und" shouldn't be added at beginning or end of span
            continue

        if root(token) != current_root:

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    
    return spans

In [12]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans2(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.5089731099521413, 'correct start': 0.8156826754691388, 'correct end': 0.6617882735405094}


In [ ]:
# punctuation and emoticons added to end of span

def get_spans3(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if token.text in [",", "."]:
            # "," & "." shouldn't be added at beginning or end of span
            continue

        if root(token) != current_root:

            # punctuation, emoticons and emojis at end of spans
            if not token.text[0].islower() and not token.text[0].isupper():
                
                # first token in a comment
                if current_root == None:
                    # create first span
                    current_span = indices
                else:
                    # add to last span
                    current_span[1] = token_end
            
                # don't change current root so that next token starts new span
                continue

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    return spans

In [14]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans3(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.567683853645925, 'correct start': 0.8133391663726239, 'correct end': 0.7148576847428221}


In [ ]:
# always split at "aber"
# not include "aber" at beginning or end of a span

def get_spans4(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if token.text == "aber":
            # start a new span
            if current_span != []:
                spans.append(current_span)
            current_span = []
            current_root = None
            comment_pos = indices[1]
            continue

        if token.text in [",", "."]:
            # "," & "." shouldn't be added at beginning or end of span
            comment_pos = indices[1]
            continue

        if root(token) != current_root:

            # punctuation, emoticons and emojis at end of spans
            if not token.text[0].islower() and not token.text[0].isupper():
                
                # first token in a comment
                if current_root == None:
                    # create first span
                    current_span = indices
                else:
                    # add to last span
                    current_span[1] = token_end
            
                # don't change current root so that next token starts new span
                comment_pos = indices[1]
                continue

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    
    return spans

In [16]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans4(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.5956658971845521, 'correct start': 0.8139482927449327, 'correct end': 0.7547012958090673}


In [ ]:
# requiring spans to continue at "und"

def get_spans5(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if token.text == "aber":
            # start a new span
            if current_span != []:
                spans.append(current_span)
            current_span = []
            current_root = None
            comment_pos = indices[1]
            continue

        if token.text == "und":
            # continue with span even if diffeent root from previous token
            if current_span == []:
                current_span = indices
            current_root = root(token)
            comment_pos = indices[1]
            continue

        if token.text in [",", "."]:
            # "," & "." shouldn't be added at beginning or end of span
            comment_pos = indices[1]
            continue

        if root(token) != current_root:

            # punctuation, emoticons and emojis at end of spans
            if not token.text[0].islower() and not token.text[0].isupper():
                
                # first token in a comment
                if current_root == None:
                    # create first span
                    current_span = indices
                else:
                    # add to last span
                    current_span[1] = token_end
            
                # don't change current root so that next token starts new span
                comment_pos = indices[1]
                continue

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    
    return spans

In [18]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans5(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.5959780339405679, 'correct start': 0.8109802394531052, 'correct end': 0.7526118822989006}


In [ ]:
# treating "und" like "," and "."
# final version

def get_spans(comment):

    current_root = None
    spans = []
    current_span = []
    comment_pos = 0

    for token in nlp(comment):
        token_start = comment_pos + comment[comment_pos:].index(token.text)
        token_end = token_start + len(token.text)
        indices = [token_start, token_end]

        if token.text == "aber":
            # start a new span
            if current_span != []:
                spans.append(current_span)
            current_span = []
            current_root = None
            comment_pos = indices[1]
            continue

        if token.text in [",", ".", "und"]:
            # ",", "." & "und" shouldn't be added at beginning or end of span
            comment_pos = indices[1]
            continue

        if root(token) != current_root:

            # punctuation, emoticons and emojis at end of spans
            if not token.text[0].islower() and not token.text[0].isupper():
                
                # first token in a comment
                if current_root == None:
                    # create first span
                    current_span = indices
                else:
                    # add to last span
                    current_span[1] = token_end
            
                # don't change current root so that next token starts new span
                comment_pos = indices[1]
                continue

            if current_span != []: # not needed for first span
                # save last span
                spans.append(current_span)

            # start new span
            current_span = indices
            current_root = root(token)
        else:
            # update end of current span
            current_span[1] = token_end
        
        comment_pos = indices[1]

    # save final span
    if current_span != []:
        spans.append(current_span)
    
    return spans

In [20]:
dependency_spans = []

for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans(train_flausch_data.iloc[i]["comment"]))

train_flausch_data["dependency_spans"] = dependency_spans

print(span_accuracy(train_flausch_data))

{'exact matches': 0.6020556874858441, 'correct start': 0.8253190595736094, 'correct end': 0.7509810322213969}


## Root dependencies

In [ ]:
def root(token):
    if token.dep_ == "ROOT":
        return token
    else:
        return root(token.head)
    

dependency_spans = []
for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans(train_flausch_data.iloc[i]["comment"]))
train_flausch_data["dependency_spans"] = dependency_spans
print(span_accuracy(train_flausch_data))

{'exact matches': 0.6015587352312602, 'correct start': 0.8004344485187846, 'correct end': 0.7645586761258986}


In [ ]:
def root(token):
    if token.dep_ in ["ROOT", "cj"]:
        return token
    else:
        return root(token.head)

dependency_spans = []
for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans(train_flausch_data.iloc[i]["comment"]))
train_flausch_data["dependency_spans"] = dependency_spans
print(span_accuracy(train_flausch_data))

{'exact matches': 0.6118569121459965, 'correct start': 0.8372735623121088, 'correct end': 0.754146168897172}


In [24]:
# best combination

def root(token):
    if token.dep_ in ["ROOT", "cd", "cj", "par", "rs"]:
        return token
    else:
        return root(token.head)

dependency_spans = []
for i in range(len(train_flausch_data)):
    dependency_spans.append(get_spans(train_flausch_data.iloc[i]["comment"]))
train_flausch_data["dependency_spans"] = dependency_spans
print(span_accuracy(train_flausch_data))

{'exact matches': 0.6121492389646878, 'correct start': 0.8401407879360413, 'correct end': 0.7527716577400522}


In [ ]:
# save span predictions for all comments on evaluation set

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(eval_all_data)):
    document = eval_all_data.iloc[i]["document"]
    comment_id = eval_all_data.iloc[i]["comment_id"]
    comment = eval_all_data.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])
eval_all_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
eval_all_spans.to_csv(path_thesis + "eval_all_spans.csv", index=False)

In [ ]:
# run classification pipeline on predicted spans for all comments

eval_all_spans = pd.read_csv(path_thesis + "eval_all_spans.csv")

# update "type" to predicted type for "text"
for i in range(len(eval_all_spans)):
    text = eval_all_spans.iloc[i]["text"]

    # update type (label)
    eval_all_spans.loc[i, "type"] = pipe(text)[0]["label"]

# save class predictions

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = eval_all_spans[eval_all_spans["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(path_thesis + "eval_all_spans_labels.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for all comments

file_gold = path_train + "gold_data_task2.csv"
file_pred = path_thesis + "eval_all_spans_labels.csv"
results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.487012987012987 0.4943968358602505 0.49067713444553485
SPANS:
  0.5207792207792208 0.5286750164798946 0.524697415767092
TYPES:
  0.7363636363636363 0.7475280158206987 0.7419038272816487
positive feedback
        STRICT     TYPES
prec  0.544643  0.795420
rec   0.525215  0.785822
f1    0.534753  0.790592

affection declaration
        STRICT     TYPES
prec  0.530612  0.778261
rec   0.515873  0.785088
f1    0.523139  0.781659

group membership
        STRICT     TYPES
prec  0.352941  0.625000
rec   0.285714  0.555556
f1    0.315789  0.588235

encouragement
        STRICT     TYPES
prec  0.320000  0.804348
rec   0.367816  0.891566
f1    0.342246  0.845714

compliment
        STRICT     TYPES
prec  0.428115  0.775281
rec   0.501873  0.873418
f1    0.462069  0.821429

ambiguous
        STRICT     TYPES
prec  0.400000  0.400000
rec   0.272727  0.272727
f1    0.324324  0.324324

implicit
        STRICT     TYPES
prec  0.125000  0.375000
rec   0.090909  0.272727
f1    0.105263  0.31

# Combination with binary classification

In [ ]:
# save span predictions for only flausch comments on evaluation set
# mimic doing binary classification before span prediction

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(eval_flausch_data)):
    document = eval_flausch_data.iloc[i]["document"]
    comment_id = eval_flausch_data.iloc[i]["comment_id"]
    comment = eval_flausch_data.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])

eval_flausch_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
eval_flausch_spans.to_csv(path_thesis + "eval_flausch_spans.csv", index=False)

In [ ]:
# run classification pipeline on predicted spans for only flausch comments

eval_flausch_spans = pd.read_csv(path_thesis + "eval_flausch_spans.csv")

# update "type" to predicted type for "text"
for i in range(len(eval_flausch_spans)):
    text = eval_flausch_spans.iloc[i]["text"]

    # update type (label)
    eval_flausch_spans.loc[i, "type"] = pipe(text)[0]["label"]

# save class predictions

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = eval_flausch_spans[eval_flausch_spans["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(path_thesis + "eval_flausch_spans_labels.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for flausch comments

file_gold = path_train + "gold_data_task2.csv"
file_pred = path_thesis + "eval_flausch_spans_labels.csv"

results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.5668934240362812 0.4943968358602505 0.528169014084507
SPANS:
  0.6061980347694633 0.5286750164798946 0.5647887323943662
TYPES:
  0.8571428571428571 0.7475280158206987 0.7985915492957747
positive feedback
        STRICT     TYPES
prec  0.627941  0.937050
rec   0.525215  0.785822
f1    0.572003  0.854799

affection declaration
        STRICT     TYPES
prec  0.601852  0.886139
rec   0.515873  0.785088
f1    0.555556  0.832558

group membership
        STRICT     TYPES
prec  0.428571  0.769231
rec   0.285714  0.555556
f1    0.342857  0.645161

encouragement
        STRICT     TYPES
prec  0.359551  0.913580
rec   0.367816  0.891566
f1    0.363636  0.902439

compliment
        STRICT     TYPES
prec  0.475177  0.873418
rec   0.501873  0.873418
f1    0.488160  0.873418

ambiguous
        STRICT     TYPES
prec  0.666667  0.666667
rec   0.272727  0.272727
f1    0.387097  0.387097

implicit
        STRICT     TYPES
prec  0.200000  0.600000
rec   0.090909  0.272727
f1    0.125000  0.37

## On test data

In [27]:
def root(token):
    if token.dep_ in ["ROOT", "cd", "cj", "par", "rs"]:
        return token
    else:
        return root(token.head)

In [ ]:
# save span predictions for all comments on test set

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(test_all_data)):
    document = test_all_data.iloc[i]["document"]
    comment_id = test_all_data.iloc[i]["comment_id"]
    comment = test_all_data.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])
test_all_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
test_all_spans.to_csv(path_thesis + "test_all_spans.csv", index=False)

In [ ]:
# run classification pipeline for all comments

test_all_spans = pd.read_csv(path_thesis + "test_all_spans.csv")

# update "type" to predicted type for "text"
for i in range(len(test_all_spans)):
    text = test_all_spans.iloc[i]["text"]

    # update type (label)
    test_all_spans.loc[i, "type"] = pipe(text)[0]["label"]

# save class predictions

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = test_all_spans[test_all_spans["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(path_thesis + "test_all_spans_labels.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for all comments

file_gold = path_test + "task2.csv"
file_pred = path_thesis + "test_all_spans_labels.csv"

results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.4588071015360064 0.38422986969595724 0.41821983816710606
SPANS:
  0.4921204867344903 0.4121282993651854 0.4485862351122829
TYPES:
  0.7496509076401356 0.627798195790177 0.6833348486226021
compliment
        STRICT     TYPES
prec  0.367647  0.674931
rec   0.410397  0.764431
f1    0.387847  0.716898

affection declaration
        STRICT     TYPES
prec  0.487928  0.823204
rec   0.399835  0.715658
f1    0.439511  0.765673

positive feedback
        STRICT     TYPES
prec  0.505238  0.829707
rec   0.419960  0.715334
f1    0.458669  0.768287

implicit
      STRICT     TYPES
prec     0.0  0.125000
rec      0.0  0.027778
f1       0.0  0.045455

ambiguous
        STRICT     TYPES
prec  0.483333  0.500000
rec   0.439394  0.454545
f1    0.460317  0.476190

encouragement
        STRICT     TYPES
prec  0.392157  0.739130
rec   0.416667  0.792373
f1    0.404040  0.764826

group membership
        STRICT     TYPES
prec  0.145455  0.730769
rec   0.020460  0.131488
f1    0.035874  0.222874



In [ ]:
# save span predictions for only flausch comments on test set

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(test_flausch_data)):
    document = test_flausch_data.iloc[i]["document"]
    comment_id = test_flausch_data.iloc[i]["comment_id"]
    comment = test_flausch_data.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])
test_flausch_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
test_flausch_spans.to_csv(path_thesis + "test_flausch_spans.csv", index=False)

In [ ]:
# run classification pipeline for only flausch comments

test_flausch_spans = pd.read_csv(path_thesis + "test_flausch_spans.csv")

# update "type" to predicted type for "text"
for i in range(len(test_flausch_spans)):
    text = test_flausch_spans.iloc[i]["text"]

    # update type (label)
    test_flausch_spans.loc[i, "type"] = pipe(text)[0]["label"]

# save class predictions

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = test_flausch_spans[test_flausch_spans["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(path_thesis + "test_flausch_spans_labels.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for only flausch comments

file_gold = path_test + "task2.csv"
file_pred = path_thesis + "test_flausch_spans_labels.csv"

results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)


typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.5163897620116749 0.38422986969595724 0.44061302681992337
SPANS:
  0.5538841490794791 0.4121282993651854 0.47260536398467434
TYPES:
  0.8437359676695105 0.627798195790177 0.7199233716475095
compliment
        STRICT     TYPES
prec  0.422535  0.785256
rec   0.410397  0.764431
f1    0.416378  0.774704

affection declaration
        STRICT     TYPES
prec  0.530055  0.898673
rec   0.399835  0.715658
f1    0.455827  0.796791

positive feedback
        STRICT     TYPES
prec  0.554867  0.922187
rec   0.419960  0.715334
f1    0.478079  0.805696

implicit
      STRICT     TYPES
prec     0.0  0.333333
rec      0.0  0.027778
f1       0.0  0.051282

ambiguous
        STRICT     TYPES
prec  0.707317  0.731707
rec   0.439394  0.454545
f1    0.542056  0.560748

encouragement
        STRICT     TYPES
prec  0.456621  0.861751
rec   0.416667  0.792373
f1    0.435730  0.825607

group membership
        STRICT     TYPES
prec  0.156863  0.791667
rec   0.020460  0.131488
f1    0.036199  0.225519


### Binary classification from Subtask 1

In [ ]:
# save span predictions for predicted flausch comments on test set

document_column = []
comment_id_column = []
type_column = []
start_column = []
end_column = []
text_column = []

for i in range(len(test_flausch_data_predicted)):
    document = test_flausch_data_predicted.iloc[i]["document"]
    comment_id = test_flausch_data_predicted.iloc[i]["comment_id"]
    comment = test_flausch_data_predicted.iloc[i]["comment"]

    spans = get_spans(comment)

    for span in spans:
        document_column.append(document)
        comment_id_column.append(comment_id)
        type_column.append("")
        start_column.append(span[0])
        end_column.append(span[1])
        text_column.append(comment[span[0]:span[1]])
test_flausch_spans = pd.DataFrame(list(zip(document_column, comment_id_column, type_column, start_column, end_column, text_column)), 
                         columns = ["document", "comment_id", "type", "start", "end", "text"])
test_flausch_spans.to_csv(path_thesis + "test_flausch_spans_pred.csv", index=False)

In [ ]:
# run classification pipeline for predicted flausch comments

test_flausch_spans = pd.read_csv(path_thesis + "test_flausch_spans_pred.csv")

# update "type" to predicted type for "text"
for i in range(len(test_flausch_spans)):
    text = test_flausch_spans.iloc[i]["text"]

    # update type (label)
    test_flausch_spans.loc[i, "type"] = pipe(text)[0]["label"]

# save class predictions

# get prediction data into right format for submission
# keep only flausch spans & remove text column
pred_data = test_flausch_spans[test_flausch_spans["type"] != "not"].drop(columns=["text"])

pred_data.to_csv(path_thesis + "test_flausch_spans_labels_pred.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
# run evaluation function for predicted flausch comments

file_gold = path_test + "task2.csv"
file_pred = path_thesis + "test_flausch_spans_labels_pred.csv"

results = fine_grained_flausch_by_label(file_gold, file_pred)

gold_data_task2 = pd.read_csv(file_gold)

typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

STRICT:
  0.48771390960947786 0.37136652188439695 0.4216616084977238
SPANS:
  0.5204036858271172 0.39625793518209157 0.4499241274658574
TYPES:
  0.8021061869240895 0.6107584363514869 0.6934749620637329
compliment
        STRICT     TYPES
prec  0.400273  0.743390
rec   0.400821  0.745710
f1    0.400547  0.744548

affection declaration
        STRICT     TYPES
prec  0.509740  0.864833
rec   0.388293  0.694524
f1    0.440805  0.770378

positive feedback
        STRICT     TYPES
prec  0.524871  0.871756
rec   0.409913  0.698613
f1    0.460323  0.775640

implicit
      STRICT     TYPES
prec     0.0  0.200000
rec      0.0  0.013889
f1       0.0  0.025974

ambiguous
        STRICT     TYPES
prec  0.630435  0.652174
rec   0.439394  0.454545
f1    0.517857  0.535714

encouragement
        STRICT     TYPES
prec  0.434389  0.835616
rec   0.400000  0.775424
f1    0.416486  0.804396

group membership
        STRICT     TYPES
prec  0.148148  0.745098
rec   0.020460  0.131488
f1    0.035955  0.223529